## Week 9 Reflection

1. Simulation: Coefficient Standard Deviation Under Heteroskedasticity

This simulation estimates the sampling standard deviation of the slope coefficient using repeated samples when errors are heteroskedastic. It then compares:

- Monte Carlo SD of $\hat\beta_1$
- Mean conventional OLS standard error
- Mean heteroskedasticity-robust standard error (HC1)
- OLS and robust SE from one representative sample

In [5]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

# Reproducibility
rng = np.random.default_rng(42)

# Data-generating process settings
n = 300               # sample size per simulation
n_sims = 2000         # number of Monte Carlo replications
beta0, beta1 = 1.0, 2.0
alpha = 0.05          # For 95% confidence intervals

# Store results across simulations
beta1_hats = np.empty(n_sims)
se_ols = np.empty(n_sims)
se_hc1 = np.empty(n_sims)
ci_ols_covers = np.empty(n_sims, dtype=bool)
ci_hc1_covers = np.empty(n_sims, dtype=bool)

# Critical value for 95% CI
t_crit = stats.t.ppf(1 - alpha / 2, df=n - 2)

for s in range(n_sims):
    # Regressor
    x = rng.uniform(0, 4, size=n)

    # Heteroskedastic errors: variance increases with x
    # eps_i ~ N(0, sigma_i^2), where sigma_i^2 = 1 + 0.8 * x_i^2
    sigma = np.sqrt(1 + 0.8 * x**2)
    eps = rng.normal(loc=0.0, scale=sigma, size=n)

    # Outcome
    y = beta0 + beta1 * x + eps

    # OLS fit
    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()
    
    b1_hat = model.params[1]
    beta1_hats[s] = b1_hat

    # Conventional OLS SE (assumes homoskedasticity)
    ols_se = model.bse[1]
    se_ols[s] = ols_se

    # Heteroskedasticity-robust SE (HC1)
    model_hc1 = model.get_robustcov_results(cov_type='HC1')
    hc1_se = model_hc1.bse[1]
    se_hc1[s] = hc1_se

    # Check CI coverage
    ci_ols_lower = b1_hat - t_crit * ols_se
    ci_ols_upper = b1_hat + t_crit * ols_se
    ci_ols_covers[s] = (ci_ols_lower <= beta1) and (beta1 <= ci_ols_upper)

    ci_hc1_lower = b1_hat - t_crit * hc1_se
    ci_hc1_upper = b1_hat + t_crit * hc1_se
    ci_hc1_covers[s] = (ci_hc1_lower <= beta1) and (beta1 <= ci_hc1_upper)


# Monte Carlo estimate of SD(beta1_hat)
mc_sd_beta1 = beta1_hats.std(ddof=1)

# Compare average model-based SE estimates across simulations
avg_se_ols = se_ols.mean()
avg_se_hc1 = se_hc1.mean()

# Calculate coverage probabilities
coverage_ols = ci_ols_covers.mean()
coverage_hc1 = ci_hc1_covers.mean()

summary = pd.DataFrame({
    'Metric': [
        'Monte Carlo SD of beta1_hat',
        'Average OLS SE (homoskedastic)',
        'Average Robust SE (HC1)',
        'Coverage of 95% OLS CI',
        'Coverage of 95% Robust CI'
    ],
    'Value': [mc_sd_beta1, avg_se_ols, avg_se_hc1, coverage_ols, coverage_hc1]
})

print('--- Simulation Summary ---')
print(f'True beta1 = {beta1}, n_sims = {n_sims}, nominal CI coverage = {1-alpha:.2f}')
print(summary.to_string(index=False, float_format=lambda v: f'{v:0.4f}'))

# Show one representative sample to mirror what you'd see in a typical OLS run
x_demo = rng.uniform(0, 4, size=n)
sigma_demo = np.sqrt(1 + 0.8 * x_demo**2)
eps_demo = rng.normal(loc=0.0, scale=sigma_demo, size=n)
y_demo = beta0 + beta1 * x_demo + eps_demo
X_demo = sm.add_constant(x_demo)

demo = sm.OLS(y_demo, X_demo).fit()
demo_hc1 = demo.get_robustcov_results(cov_type='HC1')

comparison_one_sample = pd.DataFrame({
    'Coefficient': ['const', 'x1'],
    'Estimate': demo.params,
    'OLS SE': demo.bse,
    'Robust SE (HC1)': demo_hc1.bse
})

print('\n--- One-Sample Comparison (like a standard statsmodels output) ---')
print(comparison_one_sample.to_string(index=False, float_format=lambda v: f'{v:0.4f}'))

--- Simulation Summary ---
True beta1 = 2.0, n_sims = 2000, nominal CI coverage = 0.95
                        Metric  Value
   Monte Carlo SD of beta1_hat 0.1242
Average OLS SE (homoskedastic) 0.1149
       Average Robust SE (HC1) 0.1236
        Coverage of 95% OLS CI 0.9275
     Coverage of 95% Robust CI 0.9475

--- One-Sample Comparison (like a standard statsmodels output) ---
Coefficient  Estimate  OLS SE  Robust SE (HC1)
      const    1.2111  0.2680           0.1850
         x1    1.9470  0.1137           0.1155


 The robust confidence interval achieves coverage very close to the nominal 95%, while the conventional OLS interval is under-covering because the standard errors are on average too small. This was expected because the simulated data violates the homoshedasticity assumption of OLS.

2. Simulation: Coefficient Standard Deviation Under Serial Correlation

This simulation estimates the sampling standard deviation of the slope coefficient when errors are serially correlated (non-independent). It then compares:

- The Monte Carlo standard deviation of $\hat\beta_1$.
- The average conventional OLS standard error, which incorrectly assumes independent errors.
- The average HAC (Newey-West) standard error, which is robust to autocorrelation.
- The empirical 95% confidence interval coverage for both OLS and HAC standard errors.

In [7]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

# Reproducibility
rng = np.random.default_rng(123)

# --- Simulation Settings ---
n = 300               # Sample size (time periods)
n_sims = 2000         # Number of Monte Carlo replications
beta0, beta1 = 1.0, 2.0
alpha = 0.05          # For 95% confidence intervals

# AR(1) error settings
rho = 0.8             # Autocorrelation coefficient
sigma_nu = 1.0        # SD of the white noise component of the error

# --- Storage for Simulation Results ---
beta1_hats = np.empty(n_sims)
se_ols = np.empty(n_sims)
se_hac = np.empty(n_sims)
ci_ols_covers = np.empty(n_sims, dtype=bool)
ci_hac_covers = np.empty(n_sims, dtype=bool)

# Critical value for 95% CI
t_crit = stats.t.ppf(1 - alpha / 2, df=n - 2)

for s in range(n_sims):
    # Generate regressor (e.g., a time trend)
    x = np.arange(n)

    # Generate AR(1) serially correlated errors
    eps = np.empty(n)
    eps[0] = rng.normal(loc=0, scale=sigma_nu / np.sqrt(1 - rho**2)) # Stationary distribution
    for t in range(1, n):
        nu = rng.normal(loc=0, scale=sigma_nu)
        eps[t] = rho * eps[t-1] + nu

    # Generate outcome variable
    y = beta0 + beta1 * x + eps

    # Fit OLS model
    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()

    b1_hat = model.params[1]
    beta1_hats[s] = b1_hat

    # 1. Conventional OLS standard error (incorrect)
    ols_se = model.bse[1]
    se_ols[s] = ols_se

    # 2. HAC (Newey-West) standard error (correct)
    # cov_type='HAC' with `maxlags` specified
    hac_model = model.get_robustcov_results(cov_type='HAC', maxlags=int(4*(n/100)**(2/9)))
    hac_se = hac_model.bse[1]
    se_hac[s] = hac_se

    # --- Check CI Coverage ---
    # OLS CI
    ci_ols_lower = b1_hat - t_crit * ols_se
    ci_ols_upper = b1_hat + t_crit * ols_se
    ci_ols_covers[s] = (ci_ols_lower <= beta1) and (beta1 <= ci_ols_upper)

    # HAC CI
    ci_hac_lower = b1_hat - t_crit * hac_se
    ci_hac_upper = b1_hat + t_crit * hac_se
    ci_hac_covers[s] = (ci_hac_lower <= beta1) and (beta1 <= ci_hac_upper)

# --- Analyze and Display Results ---

# Monte Carlo estimate of the true standard deviation of beta1_hat
mc_sd_beta1 = beta1_hats.std(ddof=1)

# Average standard errors from the models
avg_se_ols = se_ols.mean()
avg_se_hac = se_hac.mean()

# Calculate empirical coverage probabilities
coverage_ols = ci_ols_covers.mean()
coverage_hac = ci_hac_covers.mean()

summary_df = pd.DataFrame({
    'Metric': [
        'Monte Carlo SD of beta1_hat (True SD)',
        'Average OLS SE (Assumes Independence)',
        'Average HAC (Newey-West) SE',
        'Coverage of 95% OLS CI',
        'Coverage of 95% HAC CI'
    ],
    'Value': [mc_sd_beta1, avg_se_ols, avg_se_hac, coverage_ols, coverage_hac]
})

print('--- Simulation Summary: Serially Correlated Errors ---')
print(f'True beta1 = {beta1}, AR(1) rho = {rho}, n_sims = {n_sims}')
print(summary_df.to_string(index=False, float_format=lambda v: f'{v:0.4f}'))

# Display results from one representative sample
print('\n--- One-Sample Comparison ---')
print(f"OLS SE: {ols_se:.4f}, HAC SE: {hac_se:.4f}")

--- Simulation Summary: Serially Correlated Errors ---
True beta1 = 2.0, AR(1) rho = 0.8, n_sims = 2000
                               Metric  Value
Monte Carlo SD of beta1_hat (True SD) 0.0033
Average OLS SE (Assumes Independence) 0.0011
          Average HAC (Newey-West) SE 0.0021
               Coverage of 95% OLS CI 0.4765
               Coverage of 95% HAC CI 0.7750

--- One-Sample Comparison ---
OLS SE: 0.0010, HAC SE: 0.0017


The conventional OLS stanard error was on average less than 1/3 of the true standard deviation of the coefficient estimator, being a severe downward bias. The 95% confidence interval calculated using the OLS standard error only contains the true coefficient value ~48% of the time, which is a failure compared to the expected 95%. The HAC standard errors provide a better approximation of the true standard deviation. The 95% HAC confidence interval has a coverage rate of 78% which is a big improvement over the OLS interval. 